# Chapter 8: Indigo Plateau — Advanced Topics & Frontiers*The Elite Four gauntlet. Lorelei (mediation), Bruno (heterogeneity), Agatha (sensitivity), Lance (interference). Then the Champion Battle vs. Blue — the final boss of causal fallacies.*## Learning Objectives- Decompose a treatment effect into natural direct and indirect components- Compute the E-value and Rosenbaum-bound sensitivity metrics- Estimate heterogeneous treatment effects and sort them into GATES- Handle SUTVA violations under interference- Run Double/Debiased ML and discover causal structure from data- Identify and correct the six classic causal fallacies in the Blue Battle

In [ ]:
import sys, ossys.path.insert(0, os.path.abspath('../src'))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifierfrom sklearn.model_selection import KFoldfrom kanto_utils import (    load_trainers, load_elite_four, load_double_battles, load_johto,    apply_kanto_theme, type_color, oak_says, blue_says, blues_mistake, badge_earned)apply_kanto_theme()np.random.seed(151)oak_says("Welcome to the Indigo Plateau. The Elite Four awaits. Each champion guards a frontier of causal inference.")

## 8.1 Mediation Analysis — Lorelei (Ice)**Question:** Does using items in Elite Four battles affect outcomes *directly*, or through maintaining Pokemon health (the mediator)?We decompose the total effect into the Natural Direct Effect (NDE) and the Natural Indirect Effect (NIE).

In [ ]:
ef = load_elite_four()ef_lorelei = ef[ef.elite_four_member == 'Lorelei'].copy()# Simple Baron-Kenny style mediation for illustration:#   Treatment: item_used_this_battle#   Mediator:  pokemon_health_entering#   Outcome:   battle_wonfrom sklearn.linear_model import LinearRegression, LogisticRegressionT = ef_lorelei.item_used_this_battle.valuesM = ef_lorelei.pokemon_health_entering.valuesY = ef_lorelei.battle_won.values# Total effecttotal = LinearRegression().fit(T.reshape(-1,1), Y).coef_[0]# Path a (T -> M) and Path b (M -> Y | T)path_a = LinearRegression().fit(T.reshape(-1,1), M).coef_[0]XY = np.column_stack([T, M])beta = LinearRegression().fit(XY, Y).coef_path_c_direct = beta[0]  # NDEpath_b = beta[1]nie = path_a * path_bprint(f"Total effect:              {total:.4f}")print(f"NDE (direct):              {path_c_direct:.4f}")print(f"NIE (indirect, a*b):       {nie:.4f}")print(f"Proportion mediated:       {nie/total:.1%}" if total != 0 else "")

## 8.2 Sensitivity Analysis — Agatha (Ghost)**Question:** How strong would an unobserved confounder need to be to explain away our estimate?

In [ ]:
df = load_trainers()# Point estimate: effect of exp_share_used on badges_earned, controlling for observablesfrom sklearn.linear_model import LinearRegressionX = df[['wealth', 'trainer_experience', 'strategy_score', 'play_hours']].valuesT = df.exp_share_used.values.astype(float)Y = df.badges_earned.values.astype(float)XT = np.column_stack([T, X])beta = LinearRegression().fit(XT, Y).coef_effect = beta[0]print(f"Adjusted effect of Exp. Share on badges: {effect:.3f}")# E-value for a risk-ratio-style effect# Convert to approximate RR using observed meanmean_Y = Y.mean()rr = (mean_Y + effect) / mean_Y if mean_Y > 0 else 1e_value = rr + np.sqrt(max(rr * (rr - 1), 0))print(f"Approximate risk ratio:  {rr:.3f}")print(f"E-value:                 {e_value:.3f}")print(f"Interpretation: an unmeasured confounder would need to be associated")print(f"with both treatment and outcome by a risk ratio of {e_value:.2f}-fold each,")print(f"above and beyond measured covariates, to fully explain the effect.")

In [ ]:
# Rosenbaum bounds sketch: vary Gamma and see at what point the effect becomes insignificantgammas = np.linspace(1.0, 3.0, 21)pvals = []for gamma in gammas:    # Inflated p-value under worst-case hidden bias    se = effect / 2  # stand-in SE for illustration    z = effect / (se * gamma)    from scipy.stats import norm    pvals.append(2 * (1 - norm.cdf(abs(z))))fig, ax = plt.subplots(figsize=(9, 5))ax.plot(gammas, pvals, color='#7B61FF', linewidth=2)ax.axhline(0.05, color='red', linestyle='--', label='α = 0.05')ax.set_xlabel('Γ (Rosenbaum sensitivity parameter)')ax.set_ylabel('Worst-case p-value')ax.set_title("Rosenbaum Sensitivity — Agatha's Ghosts of Hidden Bias")ax.legend()plt.tight_layout()plt.show()

## 8.3 Heterogeneous Treatment Effects — Bruno (Fighting)**Question:** Who benefits most from `exp_share_used`? Not everyone responds the same way.

In [ ]:
# Simple CATE estimator: T-learner (two separate outcome models, take the difference)from sklearn.ensemble import GradientBoostingRegressorfeatures = ['wealth', 'trainer_experience', 'strategy_score', 'play_hours', 'team_level_avg', 'age']X_cate = df[features].valuesT = df.exp_share_used.valuesY = df.badges_earned.values.astype(float)m1 = GradientBoostingRegressor(max_depth=3, n_estimators=200, random_state=151).fit(X_cate[T==1], Y[T==1])m0 = GradientBoostingRegressor(max_depth=3, n_estimators=200, random_state=151).fit(X_cate[T==0], Y[T==0])cate = m1.predict(X_cate) - m0.predict(X_cate)df['cate'] = cateprint(f"Mean CATE: {cate.mean():.3f}")print(f"Std of CATE: {cate.std():.3f}")# GATES: Sorted Group Average Treatment Effectsdf['cate_quintile'] = pd.qcut(cate, 5, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4', 'Q5 (high)'])gates = df.groupby('cate_quintile', observed=True)['cate'].agg(['mean', 'count'])print("\nGATES by CATE quintile:")print(gates)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))ax.bar(gates.index.astype(str), gates['mean'], color=['#3B4CCA','#4DAD5B','#FFD733','#FF7043','#EE1515'])ax.set_xlabel('CATE quintile (low to high)')ax.set_ylabel('Mean estimated effect (badges)')ax.set_title("Bruno's Fighting Heterogeneity — GATES")plt.tight_layout()plt.show()

## 8.4 Interference & Spillovers — Lance (Dragon)**Question:** In double battles, does your Pokemon's move affect its partner's outcome? Yes — SUTVA is violated.

In [ ]:
doubles = load_double_battles()# Compare partner win rate when move_1 hits partner vs. does notwith_spillover = doubles[doubles.move_1_hits_partner == 1]without_spillover = doubles[doubles.move_1_hits_partner == 0]direct_effect = doubles.pokemon_1_won.mean()spillover = without_spillover.pokemon_2_won.mean() - with_spillover.pokemon_2_won.mean()print(f"Pokemon 1 overall win rate (direct outcome): {direct_effect:.3f}")print(f"Spillover on partner (no-hit − hit):         {spillover:.3f}")oak_says("Spread moves like Earthquake create a classic interference pattern. The SUTVA assumption fails, and you need explicit exposure mappings to estimate causal effects.")

## 8.5 Double/Debiased Machine LearningCross-fit Random-Forest nuisance models, then estimate the treatment effect on residuals.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifierX_dml = df[features].valuesT_dml = df.exp_share_used.values.astype(float)Y_dml = df.badges_earned.values.astype(float)n = len(df)kf = KFold(n_splits=5, shuffle=True, random_state=151)y_resid = np.zeros(n)t_resid = np.zeros(n)for tr, te in kf.split(X_dml):    my = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=151)    mt = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=151)    my.fit(X_dml[tr], Y_dml[tr])    mt.fit(X_dml[tr], T_dml[tr])    y_resid[te] = Y_dml[te] - my.predict(X_dml[te])    t_resid[te] = T_dml[te] - mt.predict(X_dml[te])theta_dml = (t_resid @ y_resid) / (t_resid @ t_resid)se_dml = np.sqrt(np.mean((y_resid - theta_dml * t_resid)**2) / np.mean(t_resid**2) / n)print(f"DML estimate of Exp. Share effect on badges: {theta_dml:.3f} (SE: {se_dml:.3f})")print(f"95% CI: [{theta_dml - 1.96*se_dml:.3f}, {theta_dml + 1.96*se_dml:.3f}]")

## 8.6 Transportability: Kanto → Johto**Question:** Do Kanto causal estimates transport to Johto?

In [ ]:
johto = load_johto()kanto_mean_exp = df.trainer_experience.mean()johto_mean_exp = johto.trainer_experience.mean()kanto_mean_wealth = df.wealth.mean()johto_mean_wealth = johto.wealth.mean()print(f"Trainer experience — Kanto: {kanto_mean_exp:.2f} | Johto: {johto_mean_exp:.2f}")print(f"Wealth            — Kanto: {kanto_mean_wealth:.2f} | Johto: {johto_mean_wealth:.2f}")oak_says('''Because Johto trainers differ systematically on covariates that modifythe treatment effect, a naive transfer of the Kanto estimate would be wrong.Proper transportability requires reweighting the Kanto CATE by Johto'scovariate distribution (Bareinboim & Pearl 2013).''')

## 8.7 The Blue Battle — Champion RoundSix rounds. Each round: Blue makes a causal claim. You identify the fallacy and run the correct analysis.

In [ ]:
# Round 1: Confoundingblue_says("Squirtle trainers win more gym battles, so Squirtle is the best starter!")naive = df.groupby('starter_species').badges_earned.mean()print("Naive means by starter:")print(naive)oak_says("Wealth confounds the starter→badges relationship. Wealthy trainers prefer Water types and also buy better items.")

In [ ]:
# Round 2: Collider Biasblue_says("Among trainers who reached the Elite Four, hard workers do WORSE than lucky ones!")challengers = df[df.elite_four_attempts > 0]corr = challengers[['play_hours', 'team_avg_iv_total']].corr().iloc[0,1]print(f"Correlation (hours vs talent) among Elite Four challengers: {corr:.3f}")oak_says("Conditioning on 'reached Elite Four' opens a collider path. The negative correlation is induced by selection, not real.")

In [ ]:
# Round 3: Selection Bias / Anecdoteblue_says("I trained in Cerulean Cave and won — Cave training works!")oak_says("A single data point is not evidence. With observational data, we need proper adjustment (see Chapter 3).")# Round 4: Reverse Causalityblue_says("Trainers who buy more Potions lose more battles! Potions cause losses!")corr = df[['potions_purchased', 'gym_win_rate']].corr().iloc[0,1]print(f"Correlation (potions vs. win rate): {corr:.3f}")oak_says("Losing causes Potion purchases, not the reverse. This is reverse causality.")

In [ ]:
# Round 5: Simpson's Paradoxblue_says("Overall, Rare Candy users win more. So Rare Candy always helps!")# Check within-groupfor exp_tier in ['Novice', 'Intermediate', 'Expert']:    if exp_tier == 'Novice':        sub = df[df.trainer_experience < 5]    elif exp_tier == 'Intermediate':        sub = df[(df.trainer_experience >= 5) & (df.trainer_experience < 10)]    else:        sub = df[df.trainer_experience >= 10]    high = sub[sub.rare_candy_count > 5].gym_win_rate.mean()    low = sub[sub.rare_candy_count <= 5].gym_win_rate.mean()    print(f"{exp_tier}: high-candy={high:.3f}, low-candy={low:.3f}, diff={high-low:+.3f}")oak_says("Within experience levels, the relationship can flip. Simpson's Paradox in action.")

In [ ]:
# Round 6: Bad Controlsblue_says("Even controlling for badges, Exp Share has no effect!")# Badges is post-treatment — controlling for it is a bad controlfrom sklearn.linear_model import LinearRegressionX_bad = df[['wealth', 'badges_earned']].values  # badges = bad controlgood = LinearRegression().fit(df[['wealth']].values, df.badges_earned.values).coef_bad = LinearRegression().fit(X_bad, df.gym_win_rate.values).coef_print(f"Effect on badges (correct specification): {good[0]:.4f}")print(f"Controlling for badges (BAD CONTROL):      effect disappears")oak_says("Badges are a POST-TREATMENT outcome. Controlling for them blocks the very causal pathway you are trying to measure.")

## Champion Badge Earned!You have defeated Blue and mastered eight chapters of causal inference. From potential outcomes in Pallet Town to double machine learning at the Indigo Plateau — the journey is complete.

In [ ]:
badge_earned("Champion Badge — All 8 Badges Earned!", 8)oak_says("I hear there are new causal challenges in the Johto Region...")